# The Analogy Task Everyone Accidentally Cheats At

**Topic:** Word embeddings &nbsp;·&nbsp; **Difficulty:** Medium

"king - man + woman = queen" is the famous demonstration, and the standard
implementation quietly excludes `king`, `man`, and `woman` from the candidate
answers. Leave them in and the nearest vector is almost always one of the three
input words. The arithmetic barely moves you, so you land back where you
started.

Reported accuracy with and without the exclusion differs by tens of points. Build
both and see.

## The data (created in the next cell)

- `E` `(400, 50)`: embeddings for 400 words, built so that several analogy
  relations hold approximately.
- `analogies` `(118, 4)`: index quadruples `(a, b, c, d)` meaning
  "`a` is to `b` as `c` is to `d`", and `d` is the answer.

## Your task

With **NumPy only**:

1. L2-normalize the embeddings.
2. For each analogy, form the query `v = E[b] - E[a] + E[c]` and rank all 400
   words by cosine similarity to `v`.
3. `acc_naive`: accuracy when the top-1 word may be any of the 400.
4. `acc_excluded`: accuracy when `a`, `b`, and `c` are removed from the
   candidate set first (the standard protocol).
5. `frac_returns_input`: the fraction of analogies where the naive top-1 is
   one of `a`, `b`, `c`.
6. Set `answers` to a dict with those three values plus `"mean_rank_excluded"`,
   the mean rank of the correct answer under the exclusion protocol (1 = best).

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks the exclusion protocol scores much higher and that the naive version
mostly returns an input word.

In [ ]:
import numpy as np

rng = np.random.default_rng(3103)
V, d = 400, 50
E = rng.normal(0, 1, (V, d))

gender = rng.normal(0, 1, d) * 0.16
plural = rng.normal(0, 1, d) * 0.16

pairs_g = [(i, i + 100) for i in range(0, 60)]
for a, b in pairs_g:
    E[b] = E[a] + gender + rng.normal(0, 0.25, d)
pairs_p = [(i + 200, i + 300) for i in range(0, 60)]
for a, b in pairs_p:
    E[b] = E[a] + plural + rng.normal(0, 0.25, d)

quads = []
for group in (pairs_g, pairs_p):
    for i in range(len(group) - 1):
        a, b = group[i]
        c, dd = group[i + 1]
        quads.append((a, b, c, dd))
analogies = np.array(quads[:120])
print(f"E {E.shape}, {len(analogies)} analogies over two relations")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("acc_naive", [], 0.05084745762711865, -0.008513797363449111, 1e-06),
    ("acc_excluded", [], 1.0, -0.16743801481449916, 1e-06),
    ("frac_returns_input", [], 0.9491525423728814, -0.15892421745105006, 1e-06),
    ("mean_rank_excluded", [], 1.0, -0.16743801481449916, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "acc_excluded" in answers:
    _gap = answers["acc_excluded"] > answers["acc_naive"] + 0.3
    _cheat = answers["frac_returns_input"] > 0.5
    print(f"  {'exclusion matters':<24} {'ok' if _gap else 'wrong'}  "
          f"(naive {answers['acc_naive']:.3f} -> excluded {answers['acc_excluded']:.3f})")
    print(f"  {'naive returns an input':<24} {'ok' if _cheat else 'wrong'}  "
          f"({answers['frac_returns_input']:.1%} of the time)")
    _ok = _ok and _gap and _cheat

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

The offset vector is small compared with the word vectors themselves, so
`E[b] - E[a] + E[c]` sits very close to `E[c]`, close enough that `c` is usually
its own nearest neighbour. Excluding the three inputs is not a trick to inflate
the score. It is what makes the task measure the *relation* rather than the
identity of `c`. Any analogy result quoted without stating the exclusion
protocol is not comparable to any other.

```python
import numpy as np

U = E / np.linalg.norm(E, axis=1, keepdims=True)

hit_naive = 0
hit_excl = 0
returns_input = 0
ranks = []

for a, b, c, dtrue in analogies:
    v = U[b] - U[a] + U[c]
    v = v / np.linalg.norm(v)
    sims = U @ v

    top_naive = int(np.argmax(sims))
    if top_naive == dtrue:
        hit_naive += 1
    if top_naive in (a, b, c):
        returns_input += 1

    s = sims.copy()
    s[[a, b, c]] = -np.inf
    order = np.argsort(-s)
    if int(order[0]) == dtrue:
        hit_excl += 1
    ranks.append(int(np.flatnonzero(order == dtrue)[0]) + 1)

n = len(analogies)
answers = {
    "acc_naive": float(hit_naive / n),
    "acc_excluded": float(hit_excl / n),
    "frac_returns_input": float(returns_input / n),
    "mean_rank_excluded": float(np.mean(ranks)),
}
```